In [16]:
!hostname

node002


In [2]:
pwd

'/work/cxiao'

In [2]:
import os
import pandas as pd
import re
import neuroboros as nb
import numpy as np
from scipy.stats import zscore
from hyperalignment import (initialize_sparse_matrix, searchlight_procrustes,
                            searchlight_weights)
from scipy.spatial.distance import cdist
from hyperalignment.local_template import compute_template
from scipy.sparse import block_diag,csr_matrix
import hyperalignment as ha
from hyperalignment.searchlight import searchlight_procrustes

In [5]:
#initialize the timing files and parameters on aligned data
from scipy.stats import gamma, zscore

TR = 1.5
n_div = 256  

tt = np.arange(20 * n_div) * TR / n_div
bold = gamma.pdf(tt, 6) - gamma.pdf(tt, 16) / 6.0
bold /= bold.sum()


conditions = [
    'positive_positive', 'positive_negative', 'positive_neutral', 
    'negative_negative', 'negative_positive', 'negative_neutral'
]


csv_path = '/work/cxiao/NEP/HA/training_run_assignments_log.csv'
input_dir = "/work/cxiao/nep_timing_re/nep_timing_wots" 

# --- 1. Load the CSV ---
if not os.path.exists(csv_path):
    print(f"ERROR: CSV file not found at {csv_path}")
else:
    log_df = pd.read_csv(csv_path)
    log_df['Subject'] = log_df['Subject'].astype(str)

    file_list = []

    print(f"Scanning for 'Parsed_Best' timing files in: {input_dir}\n")
    print("-" * 60)

    # --- 2. Extract Specific Runs ---
    for _, row in log_df.iterrows():
        sub_id = row['Subject']
        
        # Split comma-separated runs from the CSV
        raw_runs_string = str(row['Parsed_Best'])
        runs_to_process = [r.strip() for r in raw_runs_string.split(',')]
        
        for run_id in runs_to_process:
            try:
                # Expecting run_id format like "wd-01"
                task_name, run_num = run_id.split('-')
            except ValueError:
                print(f"[SKIP]    Malformed run string: {run_id} for sub-{sub_id}")
                continue
            
            # Construct the filename: sub-3102_task-wd_run-01.tsv
            expected_filename = f"sub-{sub_id}_task-{task_name}_run-{run_num}.tsv"
            file_path = os.path.join(input_dir, expected_filename)
            
            # Check if it exists
            if os.path.exists(file_path):
                print(f"[FOUND]   {expected_filename}")
                file_list.append(file_path)
            else:
                print(f"[MISSING] {expected_filename}")

    print("-" * 60)
    print(f"Total timing files successfully found: {len(file_list)}")

Scanning for 'Parsed_Best' timing files in: /work/cxiao/nep_timing_re/nep_timing_wots

------------------------------------------------------------
[FOUND]   sub-3102_task-wd_run-01.tsv
[FOUND]   sub-3102_task-wd_run-02.tsv
[FOUND]   sub-3103_task-sen_run-01.tsv
[FOUND]   sub-3103_task-wd_run-01.tsv
[FOUND]   sub-3104_task-sen_run-01.tsv
[FOUND]   sub-3104_task-wd_run-01.tsv
[FOUND]   sub-3105_task-sen_run-01.tsv
[FOUND]   sub-3105_task-wd_run-01.tsv
[FOUND]   sub-3106_task-sen_run-01.tsv
[FOUND]   sub-3106_task-wd_run-01.tsv
[FOUND]   sub-3107_task-sen_run-01.tsv
[FOUND]   sub-3107_task-wd_run-02.tsv
[FOUND]   sub-3108_task-sen_run-01.tsv
[FOUND]   sub-3108_task-wd_run-01.tsv
[FOUND]   sub-3109_task-wd_run-02.tsv
[FOUND]   sub-3109_task-sen_run-02.tsv
[FOUND]   sub-3110_task-wd_run-01.tsv
[FOUND]   sub-3110_task-wd_run-02.tsv
[FOUND]   sub-3111_task-sen_run-01.tsv
[FOUND]   sub-3111_task-wd_run-01.tsv
[FOUND]   sub-3112_task-sen_run-01.tsv
[FOUND]   sub-3112_task-wd_run-02.tsv
[FOUND]

In [9]:
#initialize the matrices

from scipy.ndimage import convolve1d

all_design_matrices = {}
kwargs = {"mode": "nearest", "origin": -len(bold) // 2}

for file_path in file_list:
    # 1. Parse Metadata
    filename = os.path.basename(file_path)
    parts = filename.replace('.tsv', '').split('_')
    
    sub_id = next((s for s in parts if 'sub-' in s), None)
    task_id = next((s for s in parts if 'task-' in s), None)
    run_id = next((s for s in parts if 'run-' in s), None)
    
    if not (sub_id and task_id and run_id):
        continue

    # 2. Set nt based on task
    task_name = task_id.split('-')[1]
    if task_name == 'wd':
        nt = 440
    elif task_name == 'sen':
        nt = 454
    else:
        continue

    # High-res time array for convolution
    t_highres = np.arange(nt * n_div) * TR / n_div
    
    # 3. Load Data
    try:
        df = pd.read_csv(file_path, sep="\t")
    except Exception:
        continue

    # 4. Generate, Convolve, and Downsample
    # List to collect processed columns for this run
    processed_columns = []
    
    for condition in conditions:
        # A. Create Boxcar
        blocks = df[df['trial_type'] == condition]
        boxcar = np.zeros_like(t_highres)
        
        for i in range(blocks.shape[0]):
            block = blocks.iloc[i]
            onset = block['onset']
            duration = block['duration']
            offset = onset + duration
            boxcar[np.logical_and(t_highres >= onset, t_highres < offset)] = 1.0
            
        # B. Convolve (Your kwargs)
        reg_convolved = convolve1d(boxcar, bold, axis=0, **kwargs)
        
        # C. Downsample (Your logic)
        # Reshape to (nt, n_div) and take the mean across the division axis
        reg_downsampled = reg_convolved.reshape(-1, n_div).mean(axis=1)
        
        processed_columns.append(reg_downsampled)

    # 5. Stack and Store
    # Stack columns: Shape becomes (nt, n_conditions)
    regressors_matrix = np.stack(processed_columns, axis=1)
    
    # Create final DataFrame indexed by TRs (not high-res time)
    unique_key = f"{sub_id}_{task_id}_{run_id}"
    all_design_matrices[unique_key] = pd.DataFrame(
        regressors_matrix, 
        columns=conditions
    )
    
    print(f"Processed {unique_key} | Shape: {regressors_matrix.shape}")

print("Done.")

Processed sub-3102_task-wd_run-01 | Shape: (440, 6)
Processed sub-3102_task-wd_run-02 | Shape: (440, 6)
Processed sub-3103_task-sen_run-01 | Shape: (454, 6)
Processed sub-3103_task-wd_run-01 | Shape: (440, 6)
Processed sub-3104_task-sen_run-01 | Shape: (454, 6)
Processed sub-3104_task-wd_run-01 | Shape: (440, 6)
Processed sub-3105_task-sen_run-01 | Shape: (454, 6)
Processed sub-3105_task-wd_run-01 | Shape: (440, 6)
Processed sub-3106_task-sen_run-01 | Shape: (454, 6)
Processed sub-3106_task-wd_run-01 | Shape: (440, 6)
Processed sub-3107_task-sen_run-01 | Shape: (454, 6)
Processed sub-3107_task-wd_run-02 | Shape: (440, 6)
Processed sub-3108_task-sen_run-01 | Shape: (454, 6)
Processed sub-3108_task-wd_run-01 | Shape: (440, 6)
Processed sub-3109_task-wd_run-02 | Shape: (440, 6)
Processed sub-3109_task-sen_run-02 | Shape: (454, 6)
Processed sub-3110_task-wd_run-01 | Shape: (440, 6)
Processed sub-3110_task-wd_run-02 | Shape: (440, 6)
Processed sub-3111_task-sen_run-01 | Shape: (454, 6)
Proc

In [20]:
#use this one
import os
import numpy as np
import pandas as pd
import neuroboros as nb
from scipy.stats import zscore
from neuroboros import smooth

# ==========================================
# 1. SETUP & DEFINITIONS
# ==========================================

full_condition_names = [
    'positive_positive', 'positive_negative', 'positive_neutral', 
    'negative_negative', 'negative_positive', 'negative_neutral'
]

full_contrasts_list = [
    [ 1,  1,  1, -1, -1, -1],  # positive vs negative prosody
    [ 1,  1, -2,  1,  1, -2],  # real vs fake
    [ 1, -1,  0, -1,  1,  0],  # positive vs negative semantics
    [ 1, -1,  0,  1, -1,  0],  # congruent vs incongruent
    [ 1,  0, -1,  1,  0, -1],  # congruent vs neutral
    [ 0,  1, -1,  0,  1, -1],  # incongruent vs neutral
    [ 1,  1,  0,  1,  1,  0],  # real vs rest
    [ 0,  0,  1,  0,  0,  1],  # fake vs rest
    [ 1,  1, -2,  0,  0,  0],  # real vs fake in positive prosody
    [ 0,  0,  0,  1,  1, -2],  # real vs fake in negative prosody
    [ 1,  1,  0,  0,  0,  0],  # real vs rest in positive prosody
    [ 0,  0,  1,  0,  0,  0],  # fake vs rest in positive prosody
    [ 0,  0,  0,  1,  1,  0],  # real vs rest in negative prosody
    [ 0,  0,  0,  0,  0,  1],  # fake vs rest in negative prosody
]

all_subjects = sorted(list(set([k.split('_')[0] for k in all_design_matrices.keys()])))
task_order = ['sen', 'wd']

confounds_dir = "/work/cxiao/NEP/nb-data/25.1.4/confounds"
output_dir = "/work/cxiao/NEP/HA/GLM/prosem_contrasts_14_smoothed10/"
aligned_data_dir = "/work/cxiao/NEP/HA/aligned"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# --- Build the smoothing matrix ONCE, outside the loop ---
FWHM = 15.0            # mm; adjust to your preferred kernel size
SPACE = 'onavg-ico32'
MASK = True           # must match the space/mask your aligned data (`dm`) is already in
M = smooth('lr', fwhm=FWHM, space=SPACE, mask=MASK)

print(f"Starting GLM analysis for {len(all_subjects)} subjects...")

# ==========================================
# 2. MAIN PROCESSING LOOP
# ==========================================

for sub_id in all_subjects:
    sid = sub_id.replace('sub-', '')
    
    for task in task_order:
        prefix = f"{sub_id}_task-{task}_run-"
        run_keys = [k for k in all_design_matrices.keys() if k.startswith(prefix)]
        run_keys.sort()
        
        for key in run_keys:
            run_str_full = key.split('_')[2]
            run_num = int(run_str_full.split('-')[1])
            
            print(f"Processing: {sub_id} | {task} | Run {run_num:02d}")
            
            try:
                # --- A. Load BOLD Data ---
                run_id = f"{task}-{run_num:02d}"
                aligned_fname = f"{sid}_{run_id}_aligned.npy"
                aligned_path = os.path.join(aligned_data_dir, aligned_fname)
                
                if not os.path.exists(aligned_path):
                    print(f"  [SKIP] Missing aligned data: {aligned_path}")
                    continue
                
                dm = np.load(aligned_path)

                # --- A2. SMOOTH the BOLD data BEFORE the GLM ---
                dm = dm @ M
                
                # --- B. Load Confounds ---
                conf_fname = f"sub-{sid}_task-{task}_run-{run_num:02d}_desc-confounds_timeseries.npy"
                conf_path = os.path.join(confounds_dir, conf_fname)
                
                if not os.path.exists(conf_path):
                    print(f"  [SKIP] Missing confound file: {conf_path}")
                    continue
                
                confounds = np.load(conf_path)
                
                # --- C. Prepare Regressors ---
                regressors_df = all_design_matrices[key]
                valid_cols_mask = (regressors_df != 0).any(axis=0)
                valid_col_names = regressors_df.columns[valid_cols_mask].tolist()
                regressors_clean = regressors_df.loc[:, valid_cols_mask].values
                
                if regressors_clean.shape[1] < 2:
                    print(f"  [SKIP] {key}: Design matrix is empty or near-empty.")
                    continue

                # --- D. Adjust Contrasts Dynamically ---
                current_run_contrasts = []
                for contrast_vec in full_contrasts_list:
                    cont_series = pd.Series(contrast_vec, index=full_condition_names)
                    new_vec = cont_series[valid_col_names].values
                    current_run_contrasts.append(new_vec)

                # --- E. Nuisance Regression (on smoothed data) ---
                beta_conf = np.linalg.lstsq(confounds, dm, rcond=None)[0]
                denoised = dm - confounds @ beta_conf
                denoised = np.nan_to_num(zscore(denoised, axis=0))
                
                # --- F. Run GLM ---
                betas, ts, R2s = nb.glm(
                    denoised, 
                    regressors_clean,
                    None,
                    contrasts=current_run_contrasts, 
                    return_r2=True
                )
                
                # --- G. Save Results ---
                save_filename = f"{sid}_{task}_{run_num:02d}.npz"
                save_path = os.path.join(output_dir, save_filename)
                nb.save(save_path, {'beta': betas, 'ts': ts, 'R2s': R2s})
                print(f"  -> Saved: {save_filename}")

            except Exception as e:
                print(f"  -> [FAILED] {key}: {e}")

print("Analysis Complete.")

Starting GLM analysis for 30 subjects...
Processing: sub-3102 | wd | Run 01
  -> Saved: 3102_wd_01.npz
Processing: sub-3102 | wd | Run 02
  -> Saved: 3102_wd_02.npz
Processing: sub-3103 | sen | Run 01
  -> Saved: 3103_sen_01.npz
Processing: sub-3103 | wd | Run 01
  -> Saved: 3103_wd_01.npz
Processing: sub-3104 | sen | Run 01
  -> Saved: 3104_sen_01.npz
Processing: sub-3104 | wd | Run 01
  -> Saved: 3104_wd_01.npz
Processing: sub-3105 | sen | Run 01
  -> Saved: 3105_sen_01.npz
Processing: sub-3105 | wd | Run 01
  -> Saved: 3105_wd_01.npz
Processing: sub-3106 | sen | Run 01
  -> Saved: 3106_sen_01.npz
Processing: sub-3106 | wd | Run 01
  -> Saved: 3106_wd_01.npz
Processing: sub-3107 | sen | Run 01
  -> Saved: 3107_sen_01.npz
Processing: sub-3107 | wd | Run 02
  -> Saved: 3107_wd_02.npz
Processing: sub-3108 | sen | Run 01
  -> Saved: 3108_sen_01.npz
Processing: sub-3108 | wd | Run 01
  -> Saved: 3108_wd_01.npz
Processing: sub-3109 | sen | Run 02
  -> Saved: 3109_sen_02.npz
Processing: sub

In [3]:
with np.load('./NEP/HA/GLM/prosem_contrasts_14_smoothed10/3102_wd_01.npz') as data:
    print(data['ts'].shape)

(14, 19341)


In [21]:
"""
Parcel-level (180 HCP-MMP parcels, bilateral) t-test analysis: Native vs L2,
using each subject's mean t-value within a parcel (averaged across both
hemispheres and all vertices in that parcel), for each contrast of interest.

Now also computes, per group per parcel: SD, SE, and 95% CI -- so the
output CSV can drive bar plots with error bars directly (no need to keep
subject-level data around separately).
"""

import os
import glob
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import neuroboros as nb

# --------------------------------------------------------------------------
# 1. Mask / parcellation (same as your existing ROI code)
# --------------------------------------------------------------------------
mask = nb.mask("lr", space="onavg-ico32")
mmp0 = np.concatenate([nb.parcellation("HCP_MMP", lr) for lr in "lr"])
mmp = mmp0[mask]   # bilateral parcel labels, values 1-180, per vertex

n_parcels = 180
parcel_ids = np.arange(1, n_parcels + 1)

# --------------------------------------------------------------------------
# 2. Contrasts to test (reuse your existing definitions)
# --------------------------------------------------------------------------
contrast_names = {
    0:  "positive_vs_negative_prosody",
    1:  "real_vs_fake",
    2:  "positive_vs_negative_semantics",
    3:  "congruent_vs_incongruent",
    4:  "congruent_vs_neutral",
    5:  "incongruent_vs_neutral",
    6:  "real_vs_rest",
    7:  "fake_vs_rest",
    8:  "real_vs_fake_in_positive",
    9:  "real_vs_fake_in_negative",
    10: "real_vs_rest_in_positive",
    11: "fake_vs_rest_in_positive",
    12: "real_vs_rest_in_negative",
    13: "fake_vs_rest_in_negative",
}
selected_contrast_idx = [6, 7, 1, 8, 9, 10, 11, 12, 13]

# --------------------------------------------------------------------------
# 3. Subjects / groups / data location (same as before)
# --------------------------------------------------------------------------
native_sids = [str(s) for s in range(3102, 3117)]
l2_sids     = [str(s) for s in range(3202, 3217)]
group_map = {sid: "Native" for sid in native_sids}
group_map.update({sid: "L2" for sid in l2_sids})
all_sids = native_sids + l2_sids

data_dir = "./NEP/HA/GLM/prosem_contrasts_14_smoothed15"
task_glob = "*"

# --------------------------------------------------------------------------
# 4. Load per-subject vertex-wise ts (all contrasts at once, more efficient
#    than reloading files per contrast)
# --------------------------------------------------------------------------
subject_ts = {}   # sid -> array (n_contrasts, n_vertices)
missing_subjects = []

for sid in all_sids:
    pattern = os.path.join(data_dir, f"{sid}_{task_glob}.npz")
    files = sorted(glob.glob(pattern))
    if len(files) == 0:
        missing_subjects.append(sid)
        continue
    ts_runs = [np.load(f, allow_pickle=True)["ts"] for f in files]
    subject_ts[sid] = np.mean(ts_runs, axis=0)   # average across runs

if missing_subjects:
    print(f"Warning: no files found for subjects: {missing_subjects}")

# --------------------------------------------------------------------------
# 5. Vertex -> parcel averaging (vectorized via bincount) + t-test per parcel
# --------------------------------------------------------------------------
def vertex_to_parcel_means(ts_vertex, mmp, n_parcels):
    """Average a vertex-wise array into per-parcel means using bincount.
    Non-parcel vertices (e.g. medial wall, labeled 0 or negative) are
    routed into bin 0 and discarded."""
    mmp_safe = np.where(mmp < 0, 0, mmp)   # <- clip negatives into a junk bin
    sums = np.bincount(mmp_safe, weights=ts_vertex, minlength=n_parcels + 1)[1:n_parcels + 1]
    counts = np.bincount(mmp_safe, minlength=n_parcels + 1)[1:n_parcels + 1]
    return sums / counts


def mean_sd_se_ci(values, ci_level=0.95):
    """Return mean, sd, se, and (ci_lower, ci_upper) for a 1D array of values."""
    n = len(values)
    mean = values.mean()
    sd = values.std(ddof=1) if n > 1 else np.nan
    se = sd / np.sqrt(n) if n > 1 else np.nan
    if n > 1:
        t_crit = stats.t.ppf((1 + ci_level) / 2, df=n - 1)
        ci_half = t_crit * se
    else:
        ci_half = np.nan
    return mean, sd, se, mean - ci_half, mean + ci_half


all_results = {}

for c_idx in selected_contrast_idx:
    c_name = contrast_names[c_idx]

    # Build subject x parcel matrix of average t-values for this contrast
    rows = []
    for sid, ts_data in subject_ts.items():
        ts_vertex = ts_data[c_idx]
        parcel_means = vertex_to_parcel_means(ts_vertex, mmp, n_parcels)
        rows.append({"sid": sid, "group": group_map[sid],
                      **{f"parcel_{pid}": parcel_means[pid - 1] for pid in parcel_ids}})
    parcel_df = pd.DataFrame(rows)

    # Two-sample t-test (Native vs L2) for each parcel
    results = []
    native_df = parcel_df[parcel_df["group"] == "Native"]
    l2_df = parcel_df[parcel_df["group"] == "L2"]

    for pid in parcel_ids:
        col = f"parcel_{pid}"
        native_vals = native_df[col].dropna()
        l2_vals = l2_df[col].dropna()
        if len(native_vals) < 2 or len(l2_vals) < 2:
            continue
        tstat, pval = stats.ttest_ind(native_vals, l2_vals, equal_var=False)

        native_mean, native_sd, native_se, native_ci_lo, native_ci_hi = mean_sd_se_ci(native_vals)
        l2_mean, l2_sd, l2_se, l2_ci_lo, l2_ci_hi = mean_sd_se_ci(l2_vals)

        results.append({
            "contrast": c_name,
            "parcel": pid,
            "t_stat": tstat,
            "p_value": pval,
            "native_mean": native_mean,
            "native_sd": native_sd,
            "native_se": native_se,
            "native_ci_lower": native_ci_lo,
            "native_ci_upper": native_ci_hi,
            "l2_mean": l2_mean,
            "l2_sd": l2_sd,
            "l2_se": l2_se,
            "l2_ci_lower": l2_ci_lo,
            "l2_ci_upper": l2_ci_hi,
            "n_native": len(native_vals),
            "n_l2": len(l2_vals),
        })

    results_df = pd.DataFrame(results)

    # FDR correction across the 180 parcel tests for this contrast
    reject, p_fdr, _, _ = multipletests(results_df["p_value"], alpha=0.05, method="fdr_bh")
    results_df["p_fdr"] = p_fdr
    results_df["significant_fdr"] = reject

    results_df = results_df.sort_values("p_value").reset_index(drop=True)
    all_results[c_name] = results_df

    out_path = os.path.join(data_dir, f"parcel_ttest_{c_name}_smoothed15.csv")
    results_df.to_csv(out_path, index=False)
    n_sig = results_df["significant_fdr"].sum()
    print(f"{c_name}: {n_sig} / {n_parcels} parcels significant (FDR < 0.05). Saved to {out_path}")

# --------------------------------------------------------------------------
# 6. Combined table across all contrasts (optional, handy for scanning)
# --------------------------------------------------------------------------
combined_df = pd.concat(all_results.values(), ignore_index=True)
combined_out_path = os.path.join(data_dir, "parcel_ttest_all_contrasts_smoothed15.csv")
combined_df.to_csv(combined_out_path, index=False)
print(f"\nSaved combined results to {combined_out_path}")

real_vs_rest: 11 / 180 parcels significant (FDR < 0.05). Saved to ./NEP/HA/GLM/prosem_contrasts_14_smoothed15/parcel_ttest_real_vs_rest_smoothed15.csv
fake_vs_rest: 11 / 180 parcels significant (FDR < 0.05). Saved to ./NEP/HA/GLM/prosem_contrasts_14_smoothed15/parcel_ttest_fake_vs_rest_smoothed15.csv
real_vs_fake: 19 / 180 parcels significant (FDR < 0.05). Saved to ./NEP/HA/GLM/prosem_contrasts_14_smoothed15/parcel_ttest_real_vs_fake_smoothed15.csv
real_vs_fake_in_positive: 10 / 180 parcels significant (FDR < 0.05). Saved to ./NEP/HA/GLM/prosem_contrasts_14_smoothed15/parcel_ttest_real_vs_fake_in_positive_smoothed15.csv
real_vs_fake_in_negative: 16 / 180 parcels significant (FDR < 0.05). Saved to ./NEP/HA/GLM/prosem_contrasts_14_smoothed15/parcel_ttest_real_vs_fake_in_negative_smoothed15.csv
real_vs_rest_in_positive: 15 / 180 parcels significant (FDR < 0.05). Saved to ./NEP/HA/GLM/prosem_contrasts_14_smoothed15/parcel_ttest_real_vs_rest_in_positive_smoothed15.csv
fake_vs_rest_in_positi

In [8]:
#Figure 2, and Fig 3 use this one

#   [ 1,  1,  1, -1, -1, -1],  #0. positive vs negative prosody
#   [ 1,  1, -2,  1,  1, -2],  #1. real vs fake
#   [ 1,  -1, 0,  -1,  1, 0],  #2. positive vs negative semantics
#   [ 1, -1,  0,  1, -1,  0],  #3. congruent vs incongruent   
#   [ 1,  0, -1,  1,  0, -1],  #4. congruent vs neutral
#   [ 0,  1,  -1,  0,  1, -1,], #5. incongruent vs neutral
#   [1, 1, 0, 1, 1, 0],  # 6.real vs rest
#   [0, 0, 1, 0, 0, 1],  # 7fake vs rest
#   [1, 1, -2, 0, 0, 0], # 8.real vs fake in positive prosody
#   [0, 0, 0, 1, 1, -2], # 9.real vs fake in negative prosody
#   [1, 1, 0, 0, 0, 0], # 10. real vs rest in positive prosody
#   [0, 0, 1, 0, 0, 0], # 11. fake  vs rest in positive prosody
#   [0, 0, 0, 1, 1, 0], # 12. real  vs rest in negative prosody
#  [0, 0, 0, 0, 0, 1], # 13.fake  vs rest in negative prosody

data_folder = './NEP/HA/GLM/prosem_contrasts_14_smoothed10/'
output_folder = './NEP/HA/GLM/prosem_contrasts_14_smoothed10/real_fake_ha'
os.makedirs(output_folder, exist_ok=True)

native_subjects = list(range(3102, 3117))
learner_subjects = list(range(3202, 3217))


TARGET_CONTRAST = 'Real > Pseudo' #change for different contrasts
CONTRAST_IDX = 9 #change for different contrasts

def load_group_data(subjects, data_folder):
    """Loads and averages all runs for a list of subjects."""
    group_data = []
    for sub in subjects:
        sub_runs = []
        for t in ['sen', 'wd']:
            for r in ['01', '02']:
                fname = os.path.join(data_folder, f"{sub}_{t}_{r}.npz")
                if os.path.exists(fname):
                    try:
                        with np.load(fname) as f:
                            sub_runs.append(f['ts'])
                    except Exception as e:
                        print(f"Warning: could not read {fname}: {e}")
        
        if sub_runs:
            group_data.append(np.mean(sub_runs, axis=0))
        else:
            print(f"Warning: No valid data found for subject {sub}")
            
    return np.array(group_data) if group_data else None

# 1. Load data
native_array = load_group_data(native_subjects, data_folder)
learner_array = load_group_data(learner_subjects, data_folder)

if native_array is None or learner_array is None:
    raise RuntimeError("Valid data missing for one or more groups.")

print(f"Loaded subjects -> Native: {len(native_array)} | Learner: {len(learner_array)}")

# 2. Isolate data specifically for the target contrast
native_c = native_array[:, CONTRAST_IDX, :]
learner_c = learner_array[:, CONTRAST_IDX, :]

# 3. Calculate means
native_mean = np.nanmean(native_c, axis=0)
learner_mean = np.nanmean(learner_c, axis=0)

# 4. Set explicit scale for the MEANS
v_min, v_max = -3, 3

# Manually load the left and right hemisphere masks for your space
lr_masks = [nb.mask(lr, 'onavg-ico32') for lr in 'lr']

# 5. Helper function to plot and save images cleanly
def plot_map(data, title, filename, **kwargs):
    print(f"Plotting: {title}")
    full_title = f"{title}"
    img = nb.plot(np.nan_to_num(data), space='onavg-ico32', cmap='RdBu_r', bar_title="$t$", mask=lr_masks, **kwargs)
    if hasattr(img, 'save'):
        img.save(os.path.join(output_folder, filename))
    return img

# 6. Generate plots using the explicit scale
img_native = plot_map(native_mean, f"Native: {TARGET_CONTRAST}", f"Native_Mean_{TARGET_CONTRAST}_np.png", vmin=v_min, vmax=v_max)
img_learner = plot_map(learner_mean, f"Learner: {TARGET_CONTRAST}", f"Learner_Mean_{TARGET_CONTRAST}_np.png", vmin=v_min, vmax=v_max)

# 7. Stack Native and Learner mean images side by side
print(f"Stacking mean images for: {TARGET_CONTRAST}")
try:
    combined_image = nb.Image.hstack([img_native, img_learner])
    save_path = os.path.join(output_folder, f"plot_combined_{TARGET_CONTRAST}_nl_np.png")
    
    if hasattr(combined_image, 'save'):
        combined_image.save(save_path)
        print(f"Successfully saved stacked image: {save_path}")
except Exception as e:
    print(f"Warning: Could not stack images. Error: {e}")

print("\nDone.")

Loaded subjects -> Native: 15 | Learner: 15
Plotting: Native: Real > Pseudo
Plotting: Learner: Real > Pseudo
Stacking mean images for: Real > Pseudo
Successfully saved stacked image: ./NEP/HA/GLM/prosem_contrasts_14_smoothed10/real_fake_ha/plot_combined_Real > Pseudo_nl_np.png

Done.


In [12]:
#fig 4, use this one, native - learner difference, in real and psedu combined


data_folder = './NEP/HA/GLM/prosem_contrasts_14_smoothed10/'
output_folder = './NEP/HA/GLM/prosem_contrasts_14_smoothed10/real_fake_ha'
os.makedirs(output_folder, exist_ok=True)

native_subjects = list(range(3102, 3117))
learner_subjects = list(range(3202, 3217))

#   [1, 1, 0, 1, 1, 0],  # 6.real vs rest
#   [0, 0, 1, 0, 0, 1],  # 7fake vs rest
#   [1, 1, 0, 0, 0, 0], # 10. real vs rest in positive prosody
#   [0, 0, 1, 0, 0, 0], # 11. fake  vs rest in positive prosody
#   [0, 0, 0, 1, 1, 0], # 12. real  vs rest in negative prosody
#  [0, 0, 0, 0, 0, 1], # 13.fake  vs rest in negative prosody

CONTRASTS = {
    12: 'Real',
    13: 'Pseudo',
}

def load_group_data(subjects, data_folder):
    """Loads and averages all runs for a list of subjects."""
    group_data = []
    for sub in subjects:
        sub_runs = []
        for t in ['sen', 'wd']:
            for r in ['01', '02']:
                fname = os.path.join(data_folder, f"{sub}_{t}_{r}.npz")
                if os.path.exists(fname):
                    try:
                        with np.load(fname) as f:
                            sub_runs.append(f['ts'])
                    except Exception as e:
                        print(f"Warning: could not read {fname}: {e}")
        
        if sub_runs:
            group_data.append(np.mean(sub_runs, axis=0))
        else:
            print(f"Warning: No valid data found for subject {sub}")
            
    return np.array(group_data) if group_data else None

# 1. Load data (once, contains all contrasts)
native_array = load_group_data(native_subjects, data_folder)
learner_array = load_group_data(learner_subjects, data_folder)

if native_array is None or learner_array is None:
    raise RuntimeError("Valid data missing for one or more groups.")

print(f"Loaded subjects -> Native: {len(native_array)} | Learner: {len(learner_array)}")

# 2. Set explicit scale for the diff maps
v_min, v_max = -3, 3

# Manually load the left and right hemisphere masks for your space
lr_masks = [nb.mask(lr, 'onavg-ico32') for lr in 'lr']

# 3. Helper function to plot and save images cleanly
def plot_map(data, title, filename, **kwargs):
    print(f"Plotting: {title}")
    full_title = f"{title}"
    img = nb.plot(np.nan_to_num(data), space='onavg-ico32', cmap='RdBu_r', bar_title="$t$", mask=lr_masks, **kwargs)
    if hasattr(img, 'save'):
        img.save(os.path.join(output_folder, filename))
    return img

# 4. Compute Native - Learner diff for each contrast, and plot
diff_imgs = []
for contrast_idx, contrast_name in CONTRASTS.items():
    native_c = native_array[:, contrast_idx, :]
    learner_c = learner_array[:, contrast_idx, :]

    native_mean = np.nanmean(native_c, axis=0)
    learner_mean = np.nanmean(learner_c, axis=0)

    diff_mean = native_mean - learner_mean

    img_diff = plot_map(
        diff_mean,
        f"Native > Learner in {contrast_name}",
        f"Diff_Native_Minus_Learner_{contrast_name}_np.png",
        vmin=v_min, vmax=v_max
    )
    diff_imgs.append(img_diff)

# 5. Combine the two difference maps (contrast 6 and contrast 7) side by side
print("Stacking diff images for contrast 6 and 7")
try:
    combined_image = nb.Image.hstack(diff_imgs)
    save_path = os.path.join(output_folder, "plot_rs_fs_combined_nl_np.png")

    if hasattr(combined_image, 'save'):
        combined_image.save(save_path)
        print(f"Successfully saved stacked image: {save_path}")
except Exception as e:
    print(f"Warning: Could not stack images. Error: {e}")

print("\nDone.")

Loaded subjects -> Native: 15 | Learner: 15
Plotting: Native > Learner in Real
Plotting: Native > Learner in Pseudo
Stacking diff images for contrast 6 and 7
Successfully saved stacked image: ./NEP/HA/GLM/prosem_contrasts_14_smoothed10/real_fake_ha/plot_rs_fs_combined_nl_np.png

Done.


In [15]:
#fig 5
data_folder = './NEP/HA/GLM/prosem_contrasts_14_smoothed10/'
output_folder = './NEP/HA/GLM/prosem_contrasts_14_smoothed10/real_fake_ha'
os.makedirs(output_folder, exist_ok=True)

# Define subject lists (reusing the sub-groups from your previous setup)
native_subjects = list(range(3102, 3117))
high_subjects = [3212,
3214,
3205,
3213,
3211,
3203,
3206,
3207]
low_subjects = [3210,
3204,
3209,
3216,
3215,
3202,
3208]


#   [ 1,  1,  1, -1, -1, -1],  #0. positive vs negative prosody
#   [ 1,  1, -2,  1,  1, -2],  #1. real vs fake
#   [ 1,  -1, 0,  -1,  1, 0],  #2. positive vs negative semantics
#   [ 1, -1,  0,  1, -1,  0],  #3. congruent vs incongruent   
#   [ 1,  0, -1,  1,  0, -1],  #4. congruent vs neutral
#   [ 0,  1,  -1,  0,  1, -1,], #5. incongruent vs neutral
#   [1, 1, 0, 1, 1, 0],  # 6.real vs rest
#   [0, 0, 1, 0, 0, 1],  # 7fake vs rest
#   [1, 1, -2, 0, 0, 0], # 8.real vs fake in positive prosody
#   [0, 0, 0, 1, 1, -2], # 9.real vs fake in negative prosody
#   [1, 1, 0, 0, 0, 0], # 10. real vs rest in positive prosody
#   [0, 0, 1, 0, 0, 0], # 11. fake  vs rest in positive prosody
#   [0, 0, 0, 1, 1, 0], # 12. real  vs rest in negative prosody
#  [0, 0, 0, 0, 0, 1], # 13.fake  vs rest in negative prosody

TARGET_CONTRAST = 'Real > Pseudo'
CONTRAST_IDX = 1

def load_group_data(subjects, data_folder):
    """Loads and averages all runs for a list of subjects."""
    group_data = []
    for sub in subjects:
        sub_runs = []
        for t in ['sen', 'wd']:
            for r in ['01', '02']:
                fname = os.path.join(data_folder, f"{sub}_{t}_{r}.npz")
                if os.path.exists(fname):
                    try:
                        with np.load(fname) as f:
                            sub_runs.append(f['ts'])
                    except Exception as e:
                        print(f"Warning: could not read {fname}: {e}")
        
        if sub_runs:
            group_data.append(np.mean(sub_runs, axis=0))
        else:
            print(f"Warning: No valid data found for subject {sub}")
            
    return np.array(group_data) if group_data else None

# Group definitions for streamlined processing
subject_groups = {
    "Native": native_subjects,
    "L2 High": high_subjects,
    "L2 Low": low_subjects,

}

# 1. Load data for all four groups
group_arrays = {}
for group_name, subjects in subject_groups.items():
    arr = load_group_data(subjects, data_folder)
    if arr is None:
        raise RuntimeError(f"Valid data missing for group: {group_name}")
    group_arrays[group_name] = arr
    print(f"Loaded subjects -> {group_name}: {len(arr)}")

# 2 & 3. Isolate data specifically for the target contrast and calculate means
group_means = {}
for group_name, arr in group_arrays.items():
    contrast_data = arr[:, CONTRAST_IDX, :]
    group_means[group_name] = np.nanmean(contrast_data, axis=0)

# 4. Set explicit scale for the MEANS
v_min, v_max = -3, 3

# Manually load the left and right hemisphere masks for your space
# (Assuming 'nb' is imported in your broader script)
lr_masks = [nb.mask(lr, 'onavg-ico32') for lr in 'lr']

# 5. Helper function to plot and save images cleanly
def plot_map(data, title, filename, **kwargs):
    print(f"Plotting: {title}")
    full_title = f"{title}"
    img = nb.plot(np.nan_to_num(data), space='onavg-ico32', cmap='RdBu_r', bar_title="$t$",  mask=lr_masks, **kwargs)
    if hasattr(img, 'save'):
        img.save(os.path.join(output_folder, filename))
    return img

# 6. Generate individual plots using the explicit scale
plot_images = []
for group_name in ["Native", "L2 High", "L2 Low"]:
    mean_data = group_means[group_name]
    title = f"{group_name}: {TARGET_CONTRAST}"
    filename = f"{group_name}_{TARGET_CONTRAST}.png"
    
    img = plot_map(mean_data, title, filename, vmin=v_min, vmax=v_max)
    plot_images.append(img)

# 7. Stack Native, Advanced, Intermediate, and Beginner means

try:
    combined_image = nb.Image.hstack(plot_images)
    save_path = os.path.join(output_folder, f"plot_combined_{TARGET_CONTRAST}_2levels.png")
    
    if hasattr(combined_image, 'save'):
        combined_image.save(save_path)
        print(f"Successfully saved stacked image: {save_path}")
except Exception as e:
    print(f"Warning: Could not stack images. Error: {e}")

print("\nDone.")

Loaded subjects -> Native: 15
Loaded subjects -> L2 High: 8
Loaded subjects -> L2 Low: 7
Plotting: Native: Real > Pseudo
Plotting: L2 High: Real > Pseudo
Plotting: L2 Low: Real > Pseudo
Successfully saved stacked image: ./NEP/HA/GLM/prosem_contrasts_14_smoothed10/real_fake_ha/plot_combined_Real > Pseudo_2levels.png

Done.


In [13]:
#recompute the contrasts for three groups, native, L2 high, L2 low

"""
Parcel-level (180 HCP-MMP parcels, bilateral) group-comparison analysis:
Native vs High-proficiency L2 vs Low-proficiency L2, using each subject's
mean t-value within a parcel (averaged across both hemispheres and all
vertices in that parcel), for each contrast of interest.

Uses one-way ANOVA (3 groups) per parcel, FDR correction across parcels,
and Tukey HSD post-hoc pairwise comparisons for parcels that survive FDR.
Also computes mean, sd, se, and 95% CI (lower/upper) for each group.
"""

import os
import glob
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import neuroboros as nb

# --------------------------------------------------------------------------
# 1. Mask / parcellation (same as your existing ROI code)
# --------------------------------------------------------------------------
mask = nb.mask("lr", space="onavg-ico32")
mmp0 = np.concatenate([nb.parcellation("HCP_MMP", lr) for lr in "lr"])
mmp = mmp0[mask]   # bilateral parcel labels, values 1-180, per vertex

n_parcels = 180
parcel_ids = np.arange(1, n_parcels + 1)

# --------------------------------------------------------------------------
# 2. Contrasts to test (reuse your existing definitions)
# --------------------------------------------------------------------------
contrast_names = {
    0:  "positive_vs_negative_prosody",
    1:  "real_vs_fake",
    2:  "positive_vs_negative_semantics",
    3:  "congruent_vs_incongruent",
    4:  "congruent_vs_neutral",
    5:  "incongruent_vs_neutral",
    6:  "real_vs_rest",
    7:  "fake_vs_rest",
    8:  "real_vs_fake_in_positive",
    9:  "real_vs_fake_in_negative",
    10: "real_vs_rest_in_positive",
    11: "fake_vs_rest_in_positive",
    12: "real_vs_rest_in_negative",
    13: "fake_vs_rest_in_negative",
}
selected_contrast_idx = [6, 7, 1, 8, 9, 10, 11, 12, 13]

# --------------------------------------------------------------------------
# 3. Subjects / groups / data location
# --------------------------------------------------------------------------
native_sids = [str(s) for s in range(3102, 3117)]  # 3102-3116

# L2 subjects split by proficiency
l2_high_sids = ["3212", "3214", "3205", "3213", "3211", "3203", "3206", "3207"]
l2_low_sids  = ["3210", "3204", "3209", "3216", "3215", "3202", "3208"]

group_map = {sid: "Native" for sid in native_sids}
group_map.update({sid: "L2_High" for sid in l2_high_sids})
group_map.update({sid: "L2_Low" for sid in l2_low_sids})

all_sids = native_sids + l2_high_sids + l2_low_sids
group_order = ["Native", "L2_High", "L2_Low"]

data_dir = "./NEP/HA/GLM/prosem_contrasts_14_smoothed10"
task_glob = "*"

# --------------------------------------------------------------------------
# 4. Load per-subject vertex-wise ts (all contrasts at once)
# --------------------------------------------------------------------------
subject_ts = {}   # sid -> array (n_contrasts, n_vertices)
missing_subjects = []

for sid in all_sids:
    pattern = os.path.join(data_dir, f"{sid}_{task_glob}.npz")
    files = sorted(glob.glob(pattern))
    if len(files) == 0:
        missing_subjects.append(sid)
        continue
    ts_runs = [np.load(f, allow_pickle=True)["ts"] for f in files]
    subject_ts[sid] = np.mean(ts_runs, axis=0)   # average across runs

if missing_subjects:
    print(f"Warning: no files found for subjects: {missing_subjects}")

# --------------------------------------------------------------------------
# 5. Vertex -> parcel averaging (vectorized via bincount)
# --------------------------------------------------------------------------
def vertex_to_parcel_means(ts_vertex, mmp, n_parcels):
    """Average a vertex-wise array into per-parcel means using bincount.
    Non-parcel vertices (e.g. medial wall, labeled 0 or negative) are
    routed into bin 0 and discarded."""
    mmp_safe = np.where(mmp < 0, 0, mmp)   # clip negatives into a junk bin
    sums = np.bincount(mmp_safe, weights=ts_vertex, minlength=n_parcels + 1)[1:n_parcels + 1]
    counts = np.bincount(mmp_safe, minlength=n_parcels + 1)[1:n_parcels + 1]
    return sums / counts

# --------------------------------------------------------------------------
# 5b. Descriptive stats helper: mean, sd, se, 95% CI via t-distribution
# --------------------------------------------------------------------------
def describe_group(vals):
    """Given a 1D array/Series of subject values, return mean, sd, se,
    and (ci_lower, ci_upper) for a 95% CI based on the t-distribution."""
    vals = np.asarray(vals, dtype=float)
    n = len(vals)
    mean = np.mean(vals)
    sd = np.std(vals, ddof=1) if n > 1 else np.nan
    se = sd / np.sqrt(n) if n > 1 else np.nan

    if n > 1:
        tcrit = stats.t.ppf(0.975, df=n - 1)
        ci_lower = mean - tcrit * se
        ci_upper = mean + tcrit * se
    else:
        ci_lower, ci_upper = np.nan, np.nan

    return mean, sd, se, ci_lower, ci_upper

# --------------------------------------------------------------------------
# 6. One-way ANOVA (3 groups) per parcel + descriptives + FDR + Tukey post-hoc
# --------------------------------------------------------------------------
all_results = {}
all_posthoc = {}

for c_idx in selected_contrast_idx:
    c_name = contrast_names[c_idx]

    # Build subject x parcel matrix of average t-values for this contrast
    rows = []
    for sid, ts_data in subject_ts.items():
        ts_vertex = ts_data[c_idx]
        parcel_means = vertex_to_parcel_means(ts_vertex, mmp, n_parcels)
        rows.append({"sid": sid, "group": group_map[sid],
                      **{f"parcel_{pid}": parcel_means[pid - 1] for pid in parcel_ids}})
    parcel_df = pd.DataFrame(rows)

    group_dfs = {g: parcel_df[parcel_df["group"] == g] for g in group_order}

    results = []
    posthoc_rows = []

    for pid in parcel_ids:
        col = f"parcel_{pid}"
        vals_by_group = {g: group_dfs[g][col].dropna() for g in group_order}

        if any(len(v) < 2 for v in vals_by_group.values()):
            continue

        fstat, pval = stats.f_oneway(*[vals_by_group[g] for g in group_order])

        # Descriptives (mean, sd, se, ci_lower, ci_upper) for each group
        desc = {}
        for g in group_order:
            mean, sd, se, ci_lower, ci_upper = describe_group(vals_by_group[g])
            g_key = g.lower()
            desc[f"{g_key}_mean"] = mean
            desc[f"{g_key}_sd"] = sd
            desc[f"{g_key}_se"] = se
            desc[f"{g_key}_ci_lower"] = ci_lower
            desc[f"{g_key}_ci_upper"] = ci_upper

        results.append({
            "contrast": c_name,
            "parcel": pid,
            "f_stat": fstat,
            "p_value": pval,
            **desc,
            "n_native": len(vals_by_group["Native"]),
            "n_l2_high": len(vals_by_group["L2_High"]),
            "n_l2_low": len(vals_by_group["L2_Low"]),
        })

    results_df = pd.DataFrame(results)

    # FDR correction across the 180 parcel tests for this contrast
    reject, p_fdr, _, _ = multipletests(results_df["p_value"], alpha=0.05, method="fdr_bh")
    results_df["p_fdr"] = p_fdr
    results_df["significant_fdr"] = reject

    results_df = results_df.sort_values("p_value").reset_index(drop=True)
    all_results[c_name] = results_df

    out_path = f"parcel_anova_{c_name}.csv"
    results_df.to_csv(out_path, index=False)
    n_sig = results_df["significant_fdr"].sum()
    print(f"{c_name}: {n_sig} / {n_parcels} parcels significant (ANOVA FDR < 0.05). Saved to {out_path}")

    # ---- Tukey HSD post-hoc for parcels that survive FDR correction ----
    sig_parcels = results_df.loc[results_df["significant_fdr"], "parcel"].tolist()
    for pid in sig_parcels:
        col = f"parcel_{pid}"
        sub = parcel_df[["group", col]].dropna()
        tukey = pairwise_tukeyhsd(endog=sub[col], groups=sub["group"], alpha=0.05)
        tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
        tukey_df.insert(0, "parcel", pid)
        tukey_df.insert(0, "contrast", c_name)
        posthoc_rows.append(tukey_df)

    if posthoc_rows:
        posthoc_df = pd.concat(posthoc_rows, ignore_index=True)
        all_posthoc[c_name] = posthoc_df
        posthoc_out_path = f"parcel_tukey_posthoc_{c_name}_smoothed10.csv"
        posthoc_df.to_csv(posthoc_out_path, index=False)
        print(f"  -> Tukey post-hoc for {len(sig_parcels)} significant parcels saved to {posthoc_out_path}")

# --------------------------------------------------------------------------
# 7. Combined tables across all contrasts (handy for scanning)
# --------------------------------------------------------------------------
combined_df = pd.concat(all_results.values(), ignore_index=True)
combined_df.to_csv("parcel_anova_all_contrasts_smoothed10.csv", index=False)
print("\nSaved combined ANOVA results to parcel_anova_all_contrasts_smoothed10.csv")

if all_posthoc:
    combined_posthoc_df = pd.concat(all_posthoc.values(), ignore_index=True)
    combined_posthoc_df.to_csv("parcel_tukey_posthoc_all_contrasts_smoothed10.csv", index=False)
    print("Saved combined Tukey post-hoc results to parcel_tukey_posthoc_all_contrasts_smoothed10.csv")

real_vs_rest: 10 / 180 parcels significant (ANOVA FDR < 0.05). Saved to parcel_anova_real_vs_rest.csv
  -> Tukey post-hoc for 10 significant parcels saved to parcel_tukey_posthoc_real_vs_rest_smoothed10.csv
fake_vs_rest: 4 / 180 parcels significant (ANOVA FDR < 0.05). Saved to parcel_anova_fake_vs_rest.csv
  -> Tukey post-hoc for 4 significant parcels saved to parcel_tukey_posthoc_fake_vs_rest_smoothed10.csv
real_vs_fake: 13 / 180 parcels significant (ANOVA FDR < 0.05). Saved to parcel_anova_real_vs_fake.csv
  -> Tukey post-hoc for 13 significant parcels saved to parcel_tukey_posthoc_real_vs_fake_smoothed10.csv
real_vs_fake_in_positive: 4 / 180 parcels significant (ANOVA FDR < 0.05). Saved to parcel_anova_real_vs_fake_in_positive.csv
  -> Tukey post-hoc for 4 significant parcels saved to parcel_tukey_posthoc_real_vs_fake_in_positive_smoothed10.csv
real_vs_fake_in_negative: 6 / 180 parcels significant (ANOVA FDR < 0.05). Saved to parcel_anova_real_vs_fake_in_negative.csv
  -> Tukey post